In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [18]:
df = pd.read_pickle("last_data.pkl")

In [19]:
df.head()

,PTF,month,day,Hour,DayOfWeek,season,Is_Holiday,Is_Weekend,Yuk_Tahmin_Plani,hour_mean,...,Smart_Block,master_segment,renewable_energy,non_renewable_energy,Lag_24,Lag_48,Lag_72,Lag_96,Lag_168,Rolling_Mean_168
2024-01-18 23:00:00,1233.61,1,Thursday,23,3,winter,0,0,37653,2526.997198,...,1_Gece_Stabıl,b,22385.52,14307.44,2193.99,2287.23,1056.99,2099.01,2146.01,2136.612798
2024-01-19 00:00:00,1232.61,1,Friday,0,4,winter,0,0,34369,2726.970967,...,1_Gece_Stabıl,b,21903.93,12133.25,2302.00,2199.99,1449.99,2149.01,2500.00,2129.068810
2024-01-19 01:00:00,1232.61,1,Friday,1,4,winter,0,0,32775,2577.374836,...,1_Gece_Stabıl,b,20525.18,11610.68,2245.00,2194.00,1248.98,1248.99,2300.01,2122.715238
2024-01-19 02:00:00,1232.61,1,Friday,2,4,winter,0,0,31604,2379.967354,...,1_Gece_Stabıl,b,19818.89,10793.08,1248.98,2138.34,1057.00,1188.88,2176.00,2117.099821
2024-01-19 03:00:00,900.00,1,Friday,3,4,winter,0,0,30888,2286.253883,...,1_Gece_Stabıl,b,19416.07,10445.28,1248.98,2099.99,1000.01,1188.88,2149.00,2109.665298


In [20]:
df.columns

Index(['PTF', 'month', 'day', 'Hour', 'DayOfWeek', 'season', 'Is_Holiday',
       'Is_Weekend', 'Yuk_Tahmin_Plani', 'hour_mean', 'month_hour_avg',
       'Smart_Block', 'master_segment', 'renewable_energy',
       'non_renewable_energy', 'Lag_24', 'Lag_48', 'Lag_72', 'Lag_96',
       'Lag_168', 'Rolling_Mean_168'],
      dtype='object')

Emtia Fiyatları (TTF, API2, Brent) - Bıst100 - VIX endeksi entegresi    

In [21]:
import yfinance as yf
import pandas as pd

# 1. Tarihleri İndeksten Dinamik Olarak Alalım
# df.index'in datetime formatında olduğunu varsayıyoruz
start_date = df.index.min().strftime('%Y-%m-%d')
# Bitiş tarihine 5 gün ekliyoruz ki 'exclusive' kuralına takılmayalım
end_date = (df.index.max() + pd.Timedelta(days=5)).strftime('%Y-%m-%d')

print(f"Veriler şu aralık için çekilecek: {start_date} - {end_date}")

# 2. Semboller (Emtia ve Finans)
tickers = {
    'Brent': 'BZ=F',       # Brent Petrol
    'TTF_Gas': 'TTF=F',    # Hollanda Gazı
    'USDTRY': 'TRY=X'      # Dolar/TL Kuru
}

# 3. Verileri Çek
print("Finansal veriler indiriliyor...")
macro_data = yf.download(list(tickers.values()), start=start_date, end=end_date)['Close']

# 4. İsimleri Düzelt
macro_data = macro_data.rename(columns={v: k for k, v in tickers.items()})

# 5. Hafta Sonlarını Doldur (Forward Fill)
macro_data = macro_data.resample('D').ffill()

# 6. KRİTİK ADIM: Lag-1 (Geleceği Görmemek İçin)
# Yarının elektriğini tahmin ederken elimizdeki son veri "Dünün Kapanışı"dır.
macro_ready = macro_data.shift(1)

# -------------------------------------------------------
# 7. ANA VERİYLE BİRLEŞTİRME (Merge)
# -------------------------------------------------------

# Birleştirme anahtarı oluşturalım (Sadece Yıl-Ay-Gün)
# Ana verinin indeksinden tarihi alıyoruz
df['Join_Date'] = df.index.date

# Macro veriden de tarihi alıyoruz
macro_ready['Join_Date'] = macro_ready.index.date

# Birleştirme (Left Join - Ana veri bozulmasın)
df = pd.merge(
    df,
    macro_ready,
    on='Join_Date',
    how='left'
)

# Geçici anahtarı silelim, kalabalık yapmasın
df.drop(columns=['Join_Date'], inplace=True)

# İndeksi tekrar düzeltelim (Merge bazen indeksi sıfırlar)
# Eğer index kaybolduysa, df.index tarih olarak kalmalıydı.
# Merge işlemi 'on' parametresiyle yapıldığında index korunmazsa, tekrar atamamız gerekebilir.
# Ama yukarıdaki yöntem satır sayısını değiştirmez, güvenlidir.

# 8. Eksik Veri Kontrolü (İlk günler boş gelebilir)
df = df.fillna(method='bfill') # Baştaki boşlukları doldur

print(df[['Brent', 'TTF_Gas', 'USDTRY']].tail())

[*********************100%***********************]  3 of 3 completed

Veriler şu aralık için çekilecek: 2024-01-18 - 2025-12-25
Finansal veriler indiriliyor...
           Brent    TTF_Gas     USDTRY
16856  60.470001  28.160999  42.785301
16857  60.470001  28.160999  42.785301
16858  60.470001  28.160999  42.785301
16859  60.470001  28.160999  42.785301
16860  60.470001  28.160999  42.785301



C:\Users\90546\AppData\Local\Temp\ipykernel_7340\1887687373.py:61: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='bfill') # Baştaki boşlukları doldur


In [28]:
df[["Brent","TTF_Gas","USDTRY"]].head()

,Brent,TTF_Gas,USDTRY
0,79.099998,27.893,30.137819
1,79.099998,27.893,30.137819
2,79.099998,27.893,30.137819
3,79.099998,27.893,30.137819
4,79.099998,27.893,30.137819


In [23]:
df

,PTF,month,day,Hour,DayOfWeek,season,Is_Holiday,Is_Weekend,Yuk_Tahmin_Plani,hour_mean,...,non_renewable_energy,Lag_24,Lag_48,Lag_72,Lag_96,Lag_168,Rolling_Mean_168,Brent,USDTRY,TTF_Gas
0,1233.61,1,Thursday,23,3,winter,0,0,37653,2526.997198,...,14307.44,2193.99,2287.23,1056.99,2099.01,2146.01,2136.612798,79.099998,30.137819,27.893000
1,1232.61,1,Friday,0,4,winter,0,0,34369,2726.970967,...,12133.25,2302.00,2199.99,1449.99,2149.01,2500.00,2129.068810,79.099998,30.137819,27.893000
2,1232.61,1,Friday,1,4,winter,0,0,32775,2577.374836,...,11610.68,2245.00,2194.00,1248.98,1248.99,2300.01,2122.715238,79.099998,30.137819,27.893000
3,1232.61,1,Friday,2,4,winter,0,0,31604,2379.967354,...,10793.08,1248.98,2138.34,1057.00,1188.88,2176.00,2117.099821,79.099998,30.137819,27.893000
4,900.00,1,Friday,3,4,winter,0,0,30888,2286.253883,...,10445.28,1248.98,2099.99,1000.01,1188.88,2149.00,2109.665298,79.099998,30.137819,27.893000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16856,3130.00,12,Saturday,7,5,winter,0,1,36463,2198.446714,...,25878.21,3400.00,3400.00,3194.99,3140.00,3117.06,2891.769107,60.470001,42.785301,28.160999
16857,3400.00,12,Saturday,8,5,winter,0,1,40663,2584.257710,...,26497.44,3400.00,3400.00,3400.00,3400.00,3400.00,2891.769107,60.470001,42.785301,28.160999
16858,2892.00,12,Saturday,9,5,winter,0,1,42510,2442.623371,...,26167.99,3400.00,3400.00,3400.00,3400.00,3368.47,2888.932976,60.470001,42.785301,28.160999
16859,2469.99,12,Saturday,10,5,winter,0,1,42566,2126.956714,...,24817.94,3349.02,3130.00,3129.01,3400.00,3077.00,2885.319821,60.470001,42.785301,28.160999


In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16861 entries, 0 to 16860
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype   
---  ------                --------------  -----   
 0   PTF                   16861 non-null  float64 
 1   month                 16861 non-null  category
 2   day                   16861 non-null  category
 3   Hour                  16861 non-null  int32   
 4   DayOfWeek             16861 non-null  int32   
 5   season                16861 non-null  category
 6   Is_Holiday            16861 non-null  category
 7   Is_Weekend            16861 non-null  category
 8   Yuk_Tahmin_Plani      16861 non-null  int64   
 9   hour_mean             16861 non-null  float64 
 10  month_hour_avg        16861 non-null  float64 
 11  Smart_Block           16861 non-null  category
 12  master_segment        16861 non-null  category
 13  renewable_energy      16861 non-null  float64 
 14  non_renewable_energy  16861 non-null  float64 
 15  La